# 🎯 Object Detection — From ONNX to Embedded C

**Renesas RA8P1 Model Zoo — Tutorial 02**

---

This notebook walks through the complete pipeline for deploying a **YOLOX-Tiny**
object detection model onto a **Renesas RA8P1 MCU** (Cortex-M85 + Ethos-U55 NPU)
using the **MERA (RUHMI) compiler**.

```
YOLOX-Tiny .pth (Megvii pretrained, COCO 80 classes)
        │
        ▼  [torch.onnx.export — download_model.py]
  ONNX FP32  (yolox_tiny_224.onnx)
        │
        ▼  [onnx2tf — download_model.py]
  TFLite FP32  (yolox_tiny_224_FP32.tflite)
        │
        ▼  [onnx2tf + TFLite PTQ — download_model.py]
  TFLite INT8  (yolox_tiny_224_INT8.tflite)
        │
        ▼  [MERA compiler — Section 4]
  C source files  (embedded_c/src_mcu/ or src_mcu_npu/)
        │
        ▼
  Flash to RA8P1 board
```

---

### Model Details

| Field | Value |
|-------|-------|
| **Model** | YOLOX-Tiny (depth=0.33, width=0.375) |
| **Task** | Object Detection (anchor-free) |
| **Dataset** | COCO 2017 — 80 classes |
| **Input** | 224×224×3 (NHWC, float [0–255], letterbox + grey pad 114) |
| **Output** | [1, 1029, 85] — 1029 predictions × (4 bbox + 1 obj + 80 cls) |
| **FPN grids** | 28×28 (stride 8) + 14×14 (stride 16) + 7×7 (stride 32) |
| **Source** | [Megvii YOLOX](https://github.com/Megvii-BaseDetection/YOLOX) |

---

### Prerequisites

> ▶️ **Run cell 2** to install all dependencies automatically.

### References
- 📖 [YOLOX Paper (arXiv)](https://arxiv.org/abs/2107.08430)
- 📖 [Megvii YOLOX GitHub](https://github.com/Megvii-BaseDetection/YOLOX)
- 📖 [COCO Dataset](https://cocodataset.org/)

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import pathlib, os

_nb_dir   = pathlib.Path(os.getcwd())
_repo     = _nb_dir.parent if _nb_dir.name == "tutorials" else _nb_dir
if not (_repo / "vision").exists():
    for _p in _nb_dir.parents:
        if (_p / "vision").exists():
            _repo = _p
            break
    else:
        raise FileNotFoundError(
            f"❌ Could not find the Model-zoo repo root.\n"
            f"   Expected a 'vision/' directory in or above: {_nb_dir}\n"
            f"   Make sure you cloned the full repo and are running this notebook from the 'tutorials/' folder."
        )

# Upgrade pip
!pip install --upgrade pip

# ── MERA (RUHMI) compiler wheel ───────────────────────────────────────────────
MERA_WHL = "mera-2.6.0+pkg.4513-cp310-cp310-manylinux_2_27_x86_64.whl"
MERA_URL  = f"https://raw.githubusercontent.com/renesas/ruhmi-framework-mcu/main/install/{MERA_WHL}"
!wget -q --show-progress -O "{MERA_WHL}" "{MERA_URL}" && pip install "{MERA_WHL}"

# # ── Model-specific requirements ───────────────────────────────────────────────
_req = _repo / "vision" / "object_detection" / "yolox_tiny" / "python" / "requirements.txt"
if not _req.exists():
    raise FileNotFoundError(f"requirements.txt not found: {_req}")
print(f"Installing from: {_req}")
!pip install -r "{_req}"


print("✅ Dependencies installed")

---
## ⚙️ Setup — Configuration

Set paths and model parameters for YOLOX-Tiny 224×224.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

NUM_CALIB_SAMPLES = 200         # calibration samples for MERA NPU quantization
SCORE_THRESH      = 0.3         # detection confidence threshold
NMS_THRESH        = 0.45        # NMS IoU threshold

FORCE_REBUILD     = True        # set True to regenerate models even if they exist

# ─────────────────────────────────────────────────────────────────────────────
import pathlib, os
import numpy as np

# Auto-detect repo root
_nb_dir   = pathlib.Path(os.getcwd())
REPO_ROOT = _nb_dir.parent if _nb_dir.name == "tutorials" else _nb_dir
if not (REPO_ROOT / "vision").exists():
    for _p in _nb_dir.parents:
        if (_p / "vision").exists():
            REPO_ROOT = _p
            break
    else:
        raise FileNotFoundError(
            f"❌ Could not find the Model-zoo repo root.\n"
            f"   Expected a 'vision/' directory in or above: {_nb_dir}\n"
            f"   Make sure you cloned the full repo and are running this notebook from the 'tutorials/' folder."
        )

OD_ROOT = REPO_ROOT / "vision" / "object_detection"

MODEL_DIR  = OD_ROOT / "yolox_tiny"
TFLITE_DIR = MODEL_DIR / "python" / "model"
EMBED_DIR  = MODEL_DIR / "embedded_c"

# Model files (pre-built, included in repo)
ONNX_PATH  = TFLITE_DIR / "yolox_tiny_224.onnx"
FP32_PATH  = TFLITE_DIR / "yolox_tiny_224_FP32.tflite"
INT8_PATH  = TFLITE_DIR / "yolox_tiny_224_INT8.tflite"

# YOLOX constants
INPUT_H, INPUT_W = 224, 224
NUM_CLASSES      = 80
PAD_VALUE        = 114
STRIDES          = (8, 16, 32)

COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train",
    "truck", "boat", "traffic light", "fire hydrant", "stop sign",
    "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep",
    "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
    "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard",
    "sports ball", "kite", "baseball bat", "baseball glove", "skateboard",
    "surfboard", "tennis racket", "bottle", "wine glass", "cup", "fork",
    "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
    "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv",
    "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave",
    "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase",
    "scissors", "teddy bear", "hair drier", "toothbrush",
]

print(f"Repo root   : {REPO_ROOT}")
print(f"Model dir   : {MODEL_DIR}")
print(f"ONNX        : {ONNX_PATH}  (exists={ONNX_PATH.exists()})")
print(f"TFLite FP32 : {FP32_PATH}  (exists={FP32_PATH.exists()})")
print(f"TFLite INT8 : {INT8_PATH}  (exists={INT8_PATH.exists()})")
print(f"Embedded C  : {EMBED_DIR}")
print(f"\n✅ YOLOX-Tiny 224×224  |  COCO 80 classes  |  Renesas RA8P1")

---
## 📥 Section 1 — Download & Build Models (Optional)

This section downloads the pretrained YOLOX-Tiny `.pth` checkpoint and runs the
full conversion pipeline:

```
yolox_tiny.pth  (Megvii GitHub release)
      │
      ▼  torch.onnx.export (via utils/yolox_model.py)
  yolox_tiny_224.onnx  (ONNX FP32)
      │
      ├──▶  onnx2tf  ──▶  yolox_tiny_224_FP32.tflite
      │
      └──▶  onnx2tf + PTQ (COCO val2017 calibration)
                       ──▶  yolox_tiny_224_INT8.tflite
```

> ⏭️ **Skip this cell** if the pre-built models already exist in `python/model/`.
> The cell checks automatically and exits early if all files are present.

### Requirements for building from scratch
- `torch`, `onnx`, `onnxsim`, `onnx2tf`, `onnxruntime`
- COCO val2017 images for calibration (auto-downloaded if not present)


In [ ]:
import pathlib, os, sys, shutil, urllib.request, zipfile

# ── Check if models already exist ─────────────────────────────────────────────
_all_exist = ONNX_PATH.exists() and FP32_PATH.exists() and INT8_PATH.exists()

if _all_exist and not FORCE_REBUILD:
    print("ℹ️  All pre-built models found — skipping download & conversion.")
    print(f"   ONNX  : {ONNX_PATH}")
    print(f"   FP32  : {FP32_PATH}")
    print(f"   INT8  : {INT8_PATH}")
    print(f"   (Set FORCE_REBUILD = True in the config cell to regenerate)")
else:
    if FORCE_REBUILD and _all_exist:
        print("🔄  FORCE_REBUILD enabled — regenerating models from scratch ...\n")
        ONNX_PATH.unlink(missing_ok=True)
        FP32_PATH.unlink(missing_ok=True)
        INT8_PATH.unlink(missing_ok=True)
    else:
        print("⚠️  One or more models missing — building from scratch ...\n")

    import torch
    from torch import nn
    import cv2
    import tensorflow as tf
    import onnx2tf

    # onnx2tf's internal test-image loader calls np.load on a pickled .npy
    # but NumPy ≥1.22 defaults allow_pickle=False → monkey-patch it
    _original_np_load = np.load
    np.load = lambda *a, **kw: _original_np_load(*a, **{**kw, "allow_pickle": True})

    try:
        from tqdm import tqdm
        _HAS_TQDM = True
    except ImportError:
        _HAS_TQDM = False

    # ── Download helper with progress bar ────────────────────────────────────
    def _download_with_progress(url, dest):
        """Download url to dest with a tqdm progress bar."""
        print(f"  Downloading {url}")
        os.makedirs(os.path.dirname(str(dest)) or ".", exist_ok=True)
        if _HAS_TQDM:
            class _Reporter:
                def __init__(self):
                    self.pbar = None
                def __call__(self, block_num, block_size, total_size):
                    if self.pbar is None:
                        self.pbar = tqdm(total=total_size, unit="B",
                                         unit_scale=True, desc="  download")
                    self.pbar.update(min(block_size, max(0, total_size - self.pbar.n)))
                    if block_num * block_size >= total_size and self.pbar:
                        self.pbar.close()
            urllib.request.urlretrieve(url, str(dest), reporthook=_Reporter())
        else:
            def _report(b, bs, total):
                sys.stdout.write(f"\r  {b*bs/1e6:.1f} / {total/1e6:.1f} MB")
                sys.stdout.flush()
            urllib.request.urlretrieve(url, str(dest), reporthook=_report)
            sys.stdout.write("\n")

    # Add the model's python dir to path for utils/yolox_model.py
    _utils_dir = str(MODEL_DIR / "python")
    if _utils_dir not in sys.path:
        sys.path.insert(0, _utils_dir)
    from utils.yolox_model import build_yolox_tiny, replace_module, SiLU

    # ── Calibration helpers ───────────────────────────────────────────────────
    _CALIB_DIR     = MODEL_DIR / "python" / "Datasets" / "val2017"
    _DATASET_DIR   = _CALIB_DIR.parent
    _COCO_VAL_URL  = "http://images.cocodataset.org/zips/val2017.zip"
    _COCO_VAL_ZIP  = _DATASET_DIR / "val2017.zip"

    def _list_images(folder, need, exts=(".jpg", ".jpeg", ".png", ".bmp")):
        if not folder.is_dir():
            return []
        out = []
        for root, _, files in os.walk(str(folder)):
            for f in sorted(files):
                if f.lower().endswith(exts):
                    out.append(os.path.join(root, f))
                    if len(out) >= need:
                        return out
        return out

    def _ensure_dataset(calib_dir):
        """Return a valid calibration directory, downloading COCO val2017 if needed."""
        calib_dir = pathlib.Path(calib_dir)
        if calib_dir.is_dir() and _list_images(calib_dir, need=1):
            print(f"[OK] Dataset found: {calib_dir}")
            return str(calib_dir)
        print(f"[!!] Dataset not found at {calib_dir}")
        _DATASET_DIR.mkdir(parents=True, exist_ok=True)
        if not _COCO_VAL_ZIP.exists():
            print(f"Downloading COCO val2017 ...")
            _download_with_progress(str(_COCO_VAL_URL), str(_COCO_VAL_ZIP))
        else:
            print(f"[OK] ZIP already present: {_COCO_VAL_ZIP}")
        print(f"Extracting {_COCO_VAL_ZIP} -> {_DATASET_DIR} ...")
        with zipfile.ZipFile(str(_COCO_VAL_ZIP), "r") as zf:
            members = zf.namelist()
            if _HAS_TQDM:
                for m in tqdm(members, desc="  extract", unit="file"):
                    zf.extract(m, path=str(_DATASET_DIR))
            else:
                for i, m in enumerate(members, 1):
                    zf.extract(m, path=str(_DATASET_DIR))
                    if i % 500 == 0 or i == len(members):
                        print(f"\r  Extracting {i}/{len(members)}", end="")
                print()
        print(f"[OK] Dataset ready: {calib_dir}")
        return str(calib_dir)

    def _preprocess_nhwc(image_path, input_size):
        """Letterbox preprocess for TFLite calibration (NHWC, float32 0-255)."""
        img = cv2.imread(image_path)
        if img is None:
            return None
        ih, iw = img.shape[:2]
        scale = min(input_size / iw, input_size / ih)
        nw, nh = int(iw * scale), int(ih * scale)
        resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
        padded = np.full((input_size, input_size, 3), PAD_VALUE, dtype=np.uint8)
        pw, ph = (input_size - nw) // 2, (input_size - nh) // 2
        padded[ph:ph + nh, pw:pw + nw] = resized
        return padded.astype(np.float32)[np.newaxis, ...]  # (1,H,W,3)

    # ── Step 1: Download .pth checkpoint ──────────────────────────────────────
    PTH_URL  = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth"
    PTH_PATH = TFLITE_DIR / "yolox_tiny.pth"
    TFLITE_DIR.mkdir(parents=True, exist_ok=True)

    if not PTH_PATH.exists():
        print(f"Downloading YOLOX-Tiny .pth ...")
        _download_with_progress(PTH_URL, PTH_PATH)
    print(f"✅ Checkpoint: {PTH_PATH}  ({PTH_PATH.stat().st_size / 1e6:.1f} MB)")

    # ── Step 2: Export to ONNX FP32 ───────────────────────────────────────────
    if not ONNX_PATH.exists():
        print(f"\nExporting PyTorch → ONNX ({INPUT_H}×{INPUT_W}) ...")
        model = build_yolox_tiny(num_classes=NUM_CLASSES)
        ckpt = torch.load(str(PTH_PATH), map_location="cpu")
        if "model" in ckpt:
            ckpt = ckpt["model"]
        model.load_state_dict(ckpt)
        model.eval()
        model = replace_module(model, nn.SiLU, SiLU)
        model.head.decode_in_inference = False

        dummy = torch.randn(1, 3, INPUT_H, INPUT_W)
        torch.onnx.export(model, dummy, str(ONNX_PATH),
                          input_names=["images"], output_names=["output"],
                          opset_version=18)

        # Simplify the ONNX graph using onnxsim.
        # torch.onnx.export often produces redundant ops (reshapes, casts, identity nodes).
        # onnxsim performs constant folding, dead-code elimination, and op fusion to
        # produce a cleaner graph — this improves onnx2tf conversion reliability downstream.
        try:
            import onnx
            from onnxsim import simplify
            m = onnx.load(str(ONNX_PATH))
            original_ir = m.ir_version
            if m.ir_version > 4:
                m.ir_version = 4
            m_simp, ok = simplify(m)
            if ok:
                m_simp.ir_version = original_ir
                onnx.save(m_simp, str(ONNX_PATH))
                print("  ONNX simplified with onnxsim")
            else:
                print("  (onnxsim check failed — keeping unsimplified)")
        except ImportError:
            print("  (onnxsim not installed — skipping)")
        except Exception as e:
            print(f"  (onnxsim failed: {e} — skipping)")

        # Merge external data into single file if needed
        try:
            import onnx as _onnx
            _m = _onnx.load(str(ONNX_PATH), load_external_data=True)
            _onnx.save_model(_m, str(ONNX_PATH), save_as_external_data=False)
            _ext = str(ONNX_PATH) + ".data"
            if os.path.exists(_ext):
                os.remove(_ext)
                print("  Merged external data into single .onnx file")
        except Exception:
            pass

        print(f"✅ ONNX: {ONNX_PATH}  ({ONNX_PATH.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"✅ ONNX already exists: {ONNX_PATH}")

    # ── Step 3: ONNX → TFLite FP32 (via onnx2tf Python API) ──────────────────
    if not FP32_PATH.exists():
        print(f"\nConverting ONNX → TFLite FP32 (via onnx2tf) ...")

        # onnx2tf looks for this exact file in CWD to skip its internal
        # pickled test-image loader (which fails with NumPy ≥1.22)
        _test_npy_name = "calibration_image_sample_data_20x128x128x3_float32.npy"
        for _dir in [str(TFLITE_DIR), "."]:
            _p = os.path.join(_dir, _test_npy_name)
            if not os.path.isfile(_p):
                np.save(_p, np.random.RandomState(0).rand(20, 128, 128, 3).astype(np.float32))

        _sm_fp32 = str(TFLITE_DIR / "saved_model_fp32")
        onnx2tf.convert(
            input_onnx_file_path=str(ONNX_PATH),
            output_folder_path=_sm_fp32,
            copy_onnx_input_output_names_to_tflite=True,
            non_verbose=True,
        )

        # Find the generated .tflite
        _fp32_candidates = []
        for root, _, files in os.walk(_sm_fp32):
            for f in files:
                if f.endswith(".tflite") and "float32" in f.lower():
                    _fp32_candidates.append(os.path.join(root, f))
        if not _fp32_candidates:
            for root, _, files in os.walk(_sm_fp32):
                for f in files:
                    if f.endswith(".tflite"):
                        _fp32_candidates.append(os.path.join(root, f))
        if not _fp32_candidates:
            raise FileNotFoundError(f"No .tflite file found in {_sm_fp32}")

        shutil.copy2(_fp32_candidates[0], str(FP32_PATH))
        shutil.rmtree(_sm_fp32, ignore_errors=True)
        # Clean up dummy npy files
        for _dir in [str(TFLITE_DIR), "."]:
            _p = os.path.join(_dir, _test_npy_name)
            if os.path.isfile(_p):
                os.remove(_p)
        print(f"✅ TFLite FP32: {FP32_PATH}  ({FP32_PATH.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"✅ TFLite FP32 already exists: {FP32_PATH}")

    # ── Step 4: ONNX → TFLite INT8 (tf.lite.TFLiteConverter + COCO calib) ────
    if not INT8_PATH.exists():
        print(f"\nConverting ONNX → TFLite INT8 (with calibration) ...")

        # Ensure calibration dataset (auto-download COCO val2017 if needed)
        _calib_dir_str = _ensure_dataset(str(_CALIB_DIR))
        _calib_paths = _list_images(pathlib.Path(_calib_dir_str), need=NUM_CALIB_SAMPLES)
        if len(_calib_paths) < 10:
            raise FileNotFoundError(
                f"Not enough calibration images in {_calib_dir_str} "
                f"(found {len(_calib_paths)}, need >= 10)."
            )
        _calib_paths = _calib_paths[:NUM_CALIB_SAMPLES]
        print(f"  Using {len(_calib_paths)} calibration images from {_calib_dir_str}")

        # onnx2tf looks for this exact file in CWD
        _test_npy_name = "calibration_image_sample_data_20x128x128x3_float32.npy"
        for _dir in [str(TFLITE_DIR), "."]:
            _p = os.path.join(_dir, _test_npy_name)
            if not os.path.isfile(_p):
                np.save(_p, np.random.RandomState(0).rand(20, 128, 128, 3).astype(np.float32))

        _sm_int8 = str(TFLITE_DIR / "saved_model_int8")
        onnx2tf.convert(
            input_onnx_file_path=str(ONNX_PATH),
            output_folder_path=_sm_int8,
            copy_onnx_input_output_names_to_tflite=True,
            non_verbose=True,
        )

        # Determine input layout
        _loaded = tf.saved_model.load(_sm_int8)
        _concrete = _loaded.signatures["serving_default"]
        _input_spec = list(_concrete.structured_input_signature[1].values())[0]
        _input_shape = _input_spec.shape
        _is_nhwc = (len(_input_shape) == 4 and _input_shape[-1] == 3)
        print(f"  SavedModel input shape: {_input_shape}")

        def _representative_dataset():
            items = _calib_paths
            if _HAS_TQDM:
                items = tqdm(items, desc="  calibrating", unit="img")
            for path in items:
                blob = _preprocess_nhwc(path, INPUT_H)
                if blob is None:
                    continue
                if not _is_nhwc:
                    blob = blob.transpose(0, 3, 1, 2)
                yield [blob.astype(np.float32)]

        # TFLite INT8 conversion
        print("  Running TFLite INT8 quantization ...")
        converter = tf.lite.TFLiteConverter.from_saved_model(_sm_int8)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = _representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

        _tflite_model = converter.convert()
        with open(str(INT8_PATH), "wb") as f:
            f.write(_tflite_model)

        shutil.rmtree(_sm_int8, ignore_errors=True)
        # Clean up dummy npy files
        for _dir in [str(TFLITE_DIR), "."]:
            _p = os.path.join(_dir, _test_npy_name)
            if os.path.isfile(_p):
                os.remove(_p)
        print(f"✅ TFLite INT8: {INT8_PATH}  ({INT8_PATH.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"✅ TFLite INT8 already exists: {INT8_PATH}")

    print(f"\n{'='*60}")
    print(f"  All models ready!")
    print(f"{'='*60}")

---
### 🔬 Section 1.1 — Verify Original PyTorch Model (Baseline Inference)

Before converting to ONNX/TFLite, run a quick inference using the **original PyTorch model**
loaded from the `.pth` checkpoint. This serves as a **ground-truth baseline** for comparing
against the ONNX, FP32 TFLite, and INT8 TFLite models later.

| What | Details |
|------|---------|
| **Model** | YOLOX-Tiny (`yolox_tiny.pth`) |
| **Framework** | PyTorch (`torch`) |
| **Input** | NCHW `[1, 3, 224, 224]`, float32, `[0–255]` (no `/255` normalization) |
| **Output** | `[1, 1029, 85]` → decode with FPN grids → NMS |


In [ ]:
import sys, cv2
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ── Load PyTorch model from .pth checkpoint ───────────────────────────────────
_utils_dir = str(MODEL_DIR / "python")
if _utils_dir not in sys.path:
    sys.path.insert(0, _utils_dir)
from utils.yolox_model import build_yolox_tiny, replace_module, SiLU

PTH_PATH = TFLITE_DIR / "yolox_tiny.pth"
print(f"PyTorch      : {torch.__version__}")
print(f"Checkpoint   : {PTH_PATH}\n")

_pt_model = build_yolox_tiny(num_classes=NUM_CLASSES)
_ckpt = torch.load(str(PTH_PATH), map_location="cpu")
if "model" in _ckpt:
    _ckpt = _ckpt["model"]
_pt_model.load_state_dict(_ckpt)
_pt_model.eval()
_pt_model = replace_module(_pt_model, nn.SiLU, SiLU)
_pt_model.head.decode_in_inference = False
print(f"✅ PyTorch model loaded  ({sum(p.numel() for p in _pt_model.parameters()):,} parameters)\n")

# ── PyTorch inference helper ──────────────────────────────────────────────────
@torch.no_grad()
def run_pytorch_yolox(model, img_bgr):
    """Run YOLOX-Tiny PyTorch inference on a BGR image (NCHW input)."""
    # Letterbox resize
    ih, iw = img_bgr.shape[:2]
    scale = min(INPUT_W / iw, INPUT_H / ih)
    new_w, new_h = int(iw * scale), int(ih * scale)
    resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    padded = np.full((INPUT_H, INPUT_W, 3), PAD_VALUE, dtype=np.uint8)
    pad_w = (INPUT_W - new_w) // 2
    pad_h = (INPUT_H - new_h) // 2
    padded[pad_h:pad_h + new_h, pad_w:pad_w + new_w] = resized

    # PyTorch expects NCHW, float32, [0–255]
    blob = padded.astype(np.float32).transpose(2, 0, 1)[np.newaxis]  # (1,3,H,W)
    tensor = torch.from_numpy(blob)

    raw = model(tensor).numpy()  # (1, 1029, 85)

    # Build grids for decoding
    all_grids, all_strides = [], []
    for s in STRIDES:
        feat_h, feat_w = INPUT_H // s, INPUT_W // s
        yy, xx = np.meshgrid(np.arange(feat_h), np.arange(feat_w), indexing="ij")
        grid = np.stack([xx.ravel(), yy.ravel()], axis=1).astype(np.float32)
        all_grids.append(grid)
        all_strides.append(np.full((feat_h * feat_w, 1), s, dtype=np.float32))
    grids = np.concatenate(all_grids, axis=0)
    exp_strides = np.concatenate(all_strides, axis=0)

    # Decode
    preds = raw[0].copy()
    preds[:, :2] = (preds[:, :2] + grids) * exp_strides
    preds[:, 2:4] = np.exp(preds[:, 2:4]) * exp_strides
    cx, cy, w, h = preds[:, 0], preds[:, 1], preds[:, 2], preds[:, 3]
    x1, y1 = cx - w / 2, cy - h / 2
    x2, y2 = cx + w / 2, cy + h / 2
    obj_conf = preds[:, 4:5]
    cls_scores = preds[:, 5:]
    cls_id = cls_scores.argmax(axis=1)
    cls_conf = cls_scores[np.arange(len(cls_id)), cls_id]
    scores = obj_conf.squeeze() * cls_conf
    dets = np.stack([x1, y1, x2, y2, scores, cls_conf, cls_id.astype(np.float32)], axis=1)

    # Filter + NMS + undo letterbox
    mask = dets[:, 4] > SCORE_THRESH
    dets = dets[mask]
    if len(dets) > 0:
        dets[:, 0] = (dets[:, 0] - pad_w) / scale
        dets[:, 1] = (dets[:, 1] - pad_h) / scale
        dets[:, 2] = (dets[:, 2] - pad_w) / scale
        dets[:, 3] = (dets[:, 3] - pad_h) / scale
        # Per-class NMS
        unique_cls = np.unique(dets[:, 6].astype(int))
        final = []
        for c in unique_cls:
            cd = dets[dets[:, 6].astype(int) == c]
            order = cd[:, 4].argsort()[::-1]
            keep = []
            while order.size > 0:
                idx = order[0]; keep.append(idx)
                if order.size == 1: break
                xx1 = np.maximum(cd[idx, 0], cd[order[1:], 0])
                yy1 = np.maximum(cd[idx, 1], cd[order[1:], 1])
                xx2 = np.minimum(cd[idx, 2], cd[order[1:], 2])
                yy2 = np.minimum(cd[idx, 3], cd[order[1:], 3])
                inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
                areas_i = (cd[idx, 2] - cd[idx, 0]) * (cd[idx, 3] - cd[idx, 1])
                areas_o = (cd[order[1:], 2] - cd[order[1:], 0]) * (cd[order[1:], 3] - cd[order[1:], 1])
                iou = inter / (areas_i + areas_o - inter + 1e-6)
                order = order[1:][iou <= NMS_THRESH]
            final.append(cd[keep])
        dets = np.concatenate(final, axis=0)
    return dets

# ── Visualisation helper ──────────────────────────────────────────────────────
def _draw_pytorch(image_rgb, results, class_names):
    palette = np.random.RandomState(42).randint(0, 256, size=(len(class_names), 3)).tolist()
    for det in results:
        x1, y1, x2, y2, score, _, cls_id = det
        cls_id = int(cls_id)
        color = tuple(palette[cls_id % len(palette)])
        cv2.rectangle(image_rgb, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        label = f"{class_names[cls_id]} {score:.2f}"
        cv2.putText(image_rgb, label, (int(x1), int(y1) - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return image_rgb

# ── Run on sample images ─────────────────────────────────────────────────────
SAMPLE_DIR = OD_ROOT / "yolox_tiny" / "python" / "sample_images"
SAMPLE_IMAGES = sorted(SAMPLE_DIR.glob("*.jpg")) + sorted(SAMPLE_DIR.glob("*.JPEG")) + sorted(SAMPLE_DIR.glob("*.png"))
if not SAMPLE_IMAGES:
    raise FileNotFoundError(f"No sample images found in {SAMPLE_DIR}")

print(f"{'='*60}")
print(f"  Original PyTorch Model — YOLOX-Tiny 224×224")
print(f"{'='*60}\n")

for img_path in tqdm(SAMPLE_IMAGES, desc="PyTorch inference", unit="img"):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        tqdm.write(f"⚠️  Could not read image: {img_path} — skipping")
        continue
    results_pt = run_pytorch_yolox(_pt_model, img_bgr)

    vis = _draw_pytorch(cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB), results_pt, COCO_CLASSES)
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    ax.imshow(vis)
    ax.set_title(f"PyTorch — {len(results_pt)} detections  ({img_path.name})")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    tqdm.write(f"  PyTorch — {len(results_pt)} detection(s):")
    for r in results_pt:
        cname = COCO_CLASSES[int(r[6])]
        tqdm.write(f"    {cname:<20s}  score={r[4]:.3f}  box=[{r[0]:.0f},{r[1]:.0f},{r[2]:.0f},{r[3]:.0f}]")
    tqdm.write("")

print(f"✅ Original PyTorch model verified — detections look correct.")
print(f"   This serves as the ground-truth baseline for ONNX & TFLite comparison.")


---
## 🔍 Section 2 — Verify Models

Verifies the models exist and inspects their I/O shapes, quantization parameters,
and file sizes.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # Force CPU — no GPU needed

import tensorflow as tf
import onnxruntime as ort

print(f"TensorFlow    : {tf.__version__}")
print(f"ONNX Runtime  : {ort.__version__}")

# ── ONNX model info ───────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  ONNX FP32 — {ONNX_PATH.name}")
print(f"{'='*55}")
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
inp_onnx = sess.get_inputs()[0]
out_onnx = sess.get_outputs()[0]
print(f"  Input  : name={inp_onnx.name!r}  shape={inp_onnx.shape}  type={inp_onnx.type}")
print(f"  Output : name={out_onnx.name!r}  shape={out_onnx.shape}  type={out_onnx.type}")
print(f"  Size   : {ONNX_PATH.stat().st_size / 1024:.1f} KB")

# ── TFLite FP32 ───────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  TFLite FP32 — {FP32_PATH.name}")
print(f"{'='*55}")
interp_fp32 = tf.lite.Interpreter(model_path=str(FP32_PATH))
interp_fp32.allocate_tensors()
inp_f = interp_fp32.get_input_details()[0]
out_f = interp_fp32.get_output_details()[0]
print(f"  Input  : shape={inp_f['shape']}  dtype={inp_f['dtype'].__name__}")
print(f"  Output : shape={out_f['shape']}  dtype={out_f['dtype'].__name__}")
print(f"  Size   : {FP32_PATH.stat().st_size / 1024:.1f} KB")

# ── TFLite INT8 ───────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  TFLite INT8 — {INT8_PATH.name}")
print(f"{'='*55}")
interp_int8 = tf.lite.Interpreter(model_path=str(INT8_PATH))
interp_int8.allocate_tensors()
inp_i = interp_int8.get_input_details()[0]
out_i = interp_int8.get_output_details()[0]
print(f"  Input  : shape={inp_i['shape']}  dtype={inp_i['dtype'].__name__}")
print(f"  Output : shape={out_i['shape']}  dtype={out_i['dtype'].__name__}")
print(f"  Input  quant: scale={inp_i['quantization_parameters']['scales'][0]:.6f}  zp={inp_i['quantization_parameters']['zero_points'][0]}")
print(f"  Output quant: scale={out_i['quantization_parameters']['scales'][0]:.6f}  zp={out_i['quantization_parameters']['zero_points'][0]}")
print(f"  Size   : {INT8_PATH.stat().st_size / 1024:.1f} KB")

print(f"\n✅ All 3 models verified")


---
## 📐 Section 3 — YOLOX Preprocessing & Postprocessing Helpers

YOLOX uses a specific preprocessing pipeline:

| Step | Description |
|------|-------------|
| 1 | Load image (BGR) |
| 2 | **Letterbox resize** to 224×224 with grey padding (114) |
| 3 | Keep float range **[0, 255]** — no `/255` normalization |
| 4 | For INT8: quantize using input scale/zp from the model |

Postprocessing decodes the anchor-free output using FPN grids:

```
Raw output [1, 1029, 85]
     │
     ▼  decode with FPN grids (strides 8, 16, 32)
  [cx, cy, w, h] → [x1, y1, x2, y2]
     │
     ▼  objectness × class_prob → confidence
     │
     ▼  score threshold + per-class NMS
     │
     ▼  undo letterbox (rescale to original image coords)
  Final detections
```


In [ ]:
import cv2
import numpy as np

# ── Grid generation (anchor-free) ─────────────────────────────────────────────
def generate_yolox_grid(input_h, input_w, strides=STRIDES):
    """Build grid offsets and stride tensors for all YOLOX FPN levels."""
    all_grids, all_strides = [], []
    for s in strides:
        feat_h, feat_w = input_h // s, input_w // s
        yy, xx = np.meshgrid(np.arange(feat_h), np.arange(feat_w), indexing="ij")
        grid = np.stack([xx.ravel(), yy.ravel()], axis=1).astype(np.float32)
        all_grids.append(grid)
        all_strides.append(np.full((feat_h * feat_w, 1), s, dtype=np.float32))
    return np.concatenate(all_grids, axis=0), np.concatenate(all_strides, axis=0)


# ── Letterbox resize ──────────────────────────────────────────────────────────
def letterbox_resize(img, target_w, target_h):
    """Resize preserving aspect ratio, pad with grey (114)."""
    ih, iw = img.shape[:2]
    scale = min(target_w / iw, target_h / ih)
    new_w, new_h = int(iw * scale), int(ih * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    padded = np.full((target_h, target_w, 3), PAD_VALUE, dtype=np.uint8)
    pad_w = (target_w - new_w) // 2
    pad_h = (target_h - new_h) // 2
    padded[pad_h:pad_h + new_h, pad_w:pad_w + new_w] = resized
    return padded, scale, (pad_w, pad_h)


# ── Decode YOLOX output ───────────────────────────────────────────────────────
def decode_yolox_output(raw_output, grids, expanded_strides):
    """Decode raw YOLOX output into [x1, y1, x2, y2, score, cls_conf, cls_id]."""
    preds = raw_output[0].copy()
    preds[:, :2] = (preds[:, :2] + grids) * expanded_strides
    preds[:, 2:4] = np.exp(preds[:, 2:4]) * expanded_strides
    cx, cy, w, h = preds[:, 0], preds[:, 1], preds[:, 2], preds[:, 3]
    x1, y1 = cx - w / 2, cy - h / 2
    x2, y2 = cx + w / 2, cy + h / 2
    obj_conf  = preds[:, 4:5]
    cls_scores = preds[:, 5:]
    cls_id   = cls_scores.argmax(axis=1)
    cls_conf = cls_scores[np.arange(len(cls_id)), cls_id]
    scores   = obj_conf.squeeze() * cls_conf
    return np.stack([x1, y1, x2, y2, scores, cls_conf, cls_id.astype(np.float32)], axis=1)


# ── NMS ────────────────────────────────────────────────────────────────────────
def nms_boxes(detections, iou_thresh=0.45):
    if len(detections) == 0:
        return detections
    x1, y1, x2, y2 = detections[:, 0], detections[:, 1], detections[:, 2], detections[:, 3]
    scores = detections[:, 4]
    areas  = (x2 - x1) * (y2 - y1)
    order  = scores.argsort()[::-1]
    keep   = []
    while order.size > 0:
        idx = order[0]
        keep.append(idx)
        if order.size == 1:
            break
        xx1 = np.maximum(x1[idx], x1[order[1:]])
        yy1 = np.maximum(y1[idx], y1[order[1:]])
        xx2 = np.minimum(x2[idx], x2[order[1:]])
        yy2 = np.minimum(y2[idx], y2[order[1:]])
        inter = np.maximum(0.0, xx2 - xx1) * np.maximum(0.0, yy2 - yy1)
        iou = inter / (areas[idx] + areas[order[1:]] - inter + 1e-6)
        order = order[1:][iou <= iou_thresh]
    return detections[keep]


# ── Full postprocess ──────────────────────────────────────────────────────────
def postprocess(detections, scale, pad, score_thresh=0.3, nms_thresh=0.45):
    """Filter by score, undo letterbox, apply per-class NMS."""
    mask = detections[:, 4] > score_thresh
    dets = detections[mask]
    if len(dets) == 0:
        return np.empty((0, 7))
    # Undo letterbox
    dets[:, 0] = (dets[:, 0] - pad[0]) / scale
    dets[:, 1] = (dets[:, 1] - pad[1]) / scale
    dets[:, 2] = (dets[:, 2] - pad[0]) / scale
    dets[:, 3] = (dets[:, 3] - pad[1]) / scale
    unique_cls = np.unique(dets[:, 6].astype(int))
    final = []
    for c in unique_cls:
        cls_dets = nms_boxes(dets[dets[:, 6].astype(int) == c], nms_thresh)
        final.append(cls_dets)
    return np.concatenate(final, axis=0) if final else np.empty((0, 7))


# ── Visualisation ─────────────────────────────────────────────────────────────
def draw_results(image, results, class_names):
    """Overlay bounding boxes and labels."""
    palette = np.random.RandomState(42).randint(0, 256, size=(len(class_names), 3)).tolist()
    for det in results:
        x1, y1, x2, y2, score, _, cls_id = det
        cls_id = int(cls_id)
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        color = tuple(palette[cls_id % len(palette)])
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        label = f"{class_names[cls_id]} {score:.2f}"
        txt_sz = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]
        cv2.rectangle(image, (x1, y1 - txt_sz[1] - 4), (x1 + txt_sz[0], y1), color, -1)
        cv2.putText(image, label, (x1, y1 - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return image


# Pre-compute grids
GRIDS, EXP_STRIDES = generate_yolox_grid(INPUT_H, INPUT_W)
print(f"FPN grid points : {len(GRIDS)}  (28×28 + 14×14 + 7×7 = {28*28}+{14*14}+{7*7})")
print(f"✅ Preprocessing & postprocessing helpers ready")

---
## 🔥 Section 4 — Quick Inference Test (TFLite FP32 vs INT8)

Run YOLOX-Tiny on a sample image using both the FP32 and INT8 TFLite models
to verify the models are working correctly before MERA compilation.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# ── Load sample images ───────────────────────────────────────────────────────
SAMPLE_DIR = OD_ROOT / "yolox_tiny" / "python" / "sample_images"
SAMPLE_IMAGES = sorted(SAMPLE_DIR.glob("*.jpg")) + sorted(SAMPLE_DIR.glob("*.JPEG")) + sorted(SAMPLE_DIR.glob("*.png"))
if not SAMPLE_IMAGES:
    raise FileNotFoundError(f"No sample images found in {SAMPLE_DIR}")
print(f"Found {len(SAMPLE_IMAGES)} sample images: {[p.name for p in SAMPLE_IMAGES]}\n")

# ── Inference helper ──────────────────────────────────────────────────────────
def run_tflite_yolox(model_path, img_bgr):
    """Run YOLOX-Tiny TFLite inference on a BGR image."""
    interp = tf.lite.Interpreter(model_path=str(model_path))
    interp.allocate_tensors()
    inp_det = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]

    padded, scale, pad = letterbox_resize(img_bgr, INPUT_W, INPUT_H)
    blob = padded.astype(np.float32)  # [0–255], no normalization

    if inp_det["dtype"] == np.int8:
        q_params = inp_det["quantization_parameters"]
        s, zp = q_params["scales"][0], q_params["zero_points"][0]
        blob = np.clip(np.round(blob / s + zp), -128, 127).astype(np.int8)

    interp.set_tensor(inp_det["index"], blob[np.newaxis])
    interp.invoke()
    raw = interp.get_tensor(out_det["index"])

    # Dequantize if needed
    if raw.dtype != np.float32:
        q_params = out_det["quantization_parameters"]
        s, zp = q_params["scales"][0], q_params["zero_points"][0]
        raw = (raw.astype(np.float32) - zp) * s

    if raw.ndim == 2:
        raw = raw[np.newaxis]

    dets = decode_yolox_output(raw, GRIDS, EXP_STRIDES)
    results = postprocess(dets, scale, pad, SCORE_THRESH, NMS_THRESH)
    return results

for img_path in tqdm(SAMPLE_IMAGES, desc="TFLite FP32/INT8", unit="img"):
    # ── Run on test image ─────────────────────────────────────────────────────────
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        tqdm.write(f"⚠️  Could not read image: {img_path} — skipping")
        continue
    results_fp32 = run_tflite_yolox(FP32_PATH, img_bgr)
    results_int8 = run_tflite_yolox(INT8_PATH, img_bgr)

    # ── Visualise ─────────────────────────────────────────────────────────────────
    vis_fp32 = draw_results(cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB), results_fp32, COCO_CLASSES)
    vis_int8 = draw_results(cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB), results_int8, COCO_CLASSES)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(vis_fp32); axes[0].set_title(f"FP32 — {len(results_fp32)} detections"); axes[0].axis("off")
    axes[1].imshow(vis_int8); axes[1].set_title(f"INT8 — {len(results_int8)} detections"); axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    # ── Print detections ──────────────────────────────────────────────────────────
    for label, res in [("FP32", results_fp32), ("INT8", results_int8)]:
        tqdm.write(f"\n{'='*50}")
        tqdm.write(f"  {label} — {len(res)} detection(s)")
        tqdm.write(f"{'='*50}")
        for r in res:
            cname = COCO_CLASSES[int(r[6])]
            tqdm.write(f"    {cname:<20s}  score={r[4]:.3f}  box=[{r[0]:.0f},{r[1]:.0f},{r[2]:.0f},{r[3]:.0f}]")


---
## 🏗️ Section 5 — MERA Compilation

Compiles the model to MCU C-code using the **MERA (RUHMI)** compiler.

---

### 5.1 — NPU Compilation

Compile for **Ethos-U55 NPU** (Requires INT8 model)

```
[CPU+NPU]  yolox_tiny_224_INT8.tflite
            │
            ▼
       python mcu_compile.py model_INT8.tflite embedded_c/src_mcu_npu/ --npu
            └── Ethos-U55 NPU C source files
```


In [ ]:
import numpy as np
import pathlib

COMPILE_PY = REPO_ROOT / "ruhmi_tools" / "mcu_compile.py"
NPU_OUT    = EMBED_DIR / "src_mcu_npu"

if not COMPILE_PY.exists():
    raise FileNotFoundError(f"❌ MERA compile script not found: {COMPILE_PY}")
if not INT8_PATH.exists():
    raise FileNotFoundError(f"❌ INT8 TFLite model not found: {INT8_PATH}\n   Run Section 1 first.")

# ── NPU compile (INT8 TFLite → Ethos-U55 NPU C) ────────────────────────────
print("=" * 60)
print("  NPU Compilation")
print("=" * 60)
print(f"  Input : {INT8_PATH}")
!python "{COMPILE_PY}" "{INT8_PATH}" "{NPU_OUT}" --npu

print(f"\n✅ NPU output → {NPU_OUT}")


---
### 5.2 — CPU-Only Compilation

Compile for **Cortex-M CPU (CMSIS-NN)**.

```
yolox_tiny_224_INT8.tflite
        │
        ▼
python mcu_compile.py model_INT8.tflite src_mcu/ --cpu
        └── CMSIS-NN C files  →  embedded_c/src_mcu/
```


In [ ]:
import shutil, os, subprocess, sys

COMPILE_PY = REPO_ROOT / "ruhmi_tools" / "mcu_compile.py"
CPU_OUT    = EMBED_DIR / "src_mcu"

if not COMPILE_PY.exists():
    raise FileNotFoundError(f"❌ MERA compile script not found: {COMPILE_PY}")
if not INT8_PATH.exists():
    raise FileNotFoundError(f"❌ INT8 TFLite model not found: {INT8_PATH}\n   Run Section 1 first.")

print("=" * 60)
print("  CPU-Only Compilation (with host evaluation)")
print("=" * 60)
print(f"  Input : {INT8_PATH}")

# ⚠️  The CMake host-evaluate build requires a C++ compiler.
# On Ubuntu 22.04, g++ may not be in PATH but g++-11 is available.
env = os.environ.copy()
if not shutil.which("g++"):
    for _ver in ["11", "12", "13", "14"]:
        _gpp = shutil.which(f"g++-{_ver}")
        _gcc = shutil.which(f"gcc-{_ver}")
        if _gpp and _gcc:
            env["CXX"] = _gpp
            env["CC"]  = _gcc
            print(f"  ⚠️  g++ not in PATH — using: CXX={_gpp}, CC={_gcc}")
            break
    else:
        print("  ❌ WARNING: No C++ compiler found! Install: sudo apt install g++-11")

# --host-evaluate builds py_compute.so via CMake so we can verify
# the compiled model on the host before flashing to the board.
result = subprocess.run(
    [sys.executable, str(COMPILE_PY), str(INT8_PATH), str(CPU_OUT), "--cpu", "--host-evaluate"],
    env=env,
)

print(f"\n✅ CPU output → {CPU_OUT}")
if result.returncode != 0:
    print(f"  ⚠️  mcu_compile exited with code {result.returncode}")

---
## ✅ Section 6 — Verify MERA-Compiled Model (py_compute Inference)

Since we used `--host-evaluate` in Section 5.2, the `mcu_compile.py` script has
already built the MERA-generated C source into a `py_compute` shared library
(via CMake with `-DBUILD_PY_BINDINGS=ON`).

This section loads the pre-built `py_compute.so` from `embedded_c/src_mcu/build/`
and runs inference on sample images to visually verify that the compiled model
produces correct detections — matching the TFLite INT8 results from Section 4.

> The `py_compute` library is a Python-callable wrapper around the same C-code
> that will run on the RA8P1 board, so matching results here confirms the
> MERA compilation preserved model accuracy.

### ⚠️ Host Build Requirements (for `--host-evaluate`)

The `--host-evaluate` flag in Section 5.2 compiles the MERA-generated C source on
your host machine using CMake. If the build fails, check these requirements:

| Requirement | Minimum Version | Check Command | Install / Fix |
|-------------|----------------|---------------|---------------|
| **CMake** | ≥ 3.24 | `cmake --version` | `pip install --upgrade cmake` |
| **C++ compiler** (g++) | g++-11 recommended | `g++ --version` | `sudo apt install g++-11` |
| **build-essential** | — | `dpkg -l build-essential` | `sudo apt install build-essential` |

> 💡 **Common issues:**
> - **CMake too old** — Ubuntu 22.04 ships CMake 3.22, but the build needs ≥ 3.24.
>   Upgrade via pip: `pip install --upgrade cmake`
> - **g++ not in PATH** — If only `g++-11` is installed (not `g++`), set the compiler
>   before running Section 5.2:
>   ```bash
>   export CXX=/usr/bin/g++-11
>   export CC=/usr/bin/gcc-11
>   ```
> - **pybind11** is fetched automatically by CMake — no manual install needed.

In [ ]:
import subprocess, pickle
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ── Locate the pre-built py_compute.so ────────────────────────────────────────
CPU_BUILD_DIR = EMBED_DIR / "src_mcu"
py_compute_candidates = (
    list(CPU_BUILD_DIR.rglob("py_compute*.so"))
    + list(CPU_BUILD_DIR.rglob("py_compute*.pyd"))
)

if not py_compute_candidates:
    raise FileNotFoundError(
        f"❌ py_compute shared library not found in {CPU_BUILD_DIR}\n"
        f"   Make sure Section 5.2 ran with --host-evaluate."
    )

py_compute_dir = str(py_compute_candidates[0].parent)
print(f"✅ Found py_compute: {py_compute_candidates[0]}")

# ── Read INT8 quantization parameters from the TFLite model ──────────────────
interp_q = tf.lite.Interpreter(model_path=str(INT8_PATH))
interp_q.allocate_tensors()
_inp_det = interp_q.get_input_details()[0]
_out_det = interp_q.get_output_details()[0]
_inp_qp = _inp_det["quantization_parameters"]
_out_qp = _out_det["quantization_parameters"]
quantized_io = {
    "input_scale":  float(_inp_qp["scales"][0]),
    "input_zp":     int(_inp_qp["zero_points"][0]),
    "output_scale": float(_out_qp["scales"][0]),
    "output_zp":    int(_out_qp["zero_points"][0]),
}
del interp_q
print(f"INT8 quantization params:")
print(f"  Input  : scale={quantized_io['input_scale']:.6f}  zp={quantized_io['input_zp']}")
print(f"  Output : scale={quantized_io['output_scale']:.6f}  zp={quantized_io['output_zp']}")

# ── Subprocess inference (avoids symbol conflicts in main process) ────────────
def run_mera_inference_subprocess(py_compute_dir, sample_images, quantized_io):
    """Run MERA py_compute inference in a subprocess on sample images."""
    import sys

    config_file = "/tmp/mera_yolox_nb_config.pkl"
    result_file = "/tmp/mera_yolox_nb_results.pkl"

    config = {
        "py_compute_dir": py_compute_dir,
        "sample_images": [str(p) for p in sample_images],
        "input_h": INPUT_H, "input_w": INPUT_W,
        "strides": STRIDES, "pad_value": PAD_VALUE,
        "score_thresh": SCORE_THRESH, "nms_thresh": NMS_THRESH,
        "quantized_io": quantized_io,
    }
    with open(config_file, "wb") as f:
        pickle.dump(config, f)

    script = f'''
import sys, os, pickle
import numpy as np, cv2

with open("{config_file}", "rb") as f:
    cfg = pickle.load(f)

def generate_yolox_grid(input_h, input_w, strides):
    all_grids, all_strides = [], []
    for s in strides:
        fh, fw = input_h // s, input_w // s
        yy, xx = np.meshgrid(np.arange(fh), np.arange(fw), indexing="ij")
        all_grids.append(np.stack([xx.ravel(), yy.ravel()], 1).astype(np.float32))
        all_strides.append(np.full((fh * fw, 1), s, dtype=np.float32))
    return np.concatenate(all_grids, 0), np.concatenate(all_strides, 0)

def letterbox_resize(img, tw, th, pv):
    ih, iw = img.shape[:2]
    sc = min(tw / iw, th / ih); nw, nh = int(iw * sc), int(ih * sc)
    resized = cv2.resize(img, (nw, nh))
    padded = np.full((th, tw, 3), pv, dtype=np.uint8)
    pw, ph = (tw - nw) // 2, (th - nh) // 2
    padded[ph:ph+nh, pw:pw+nw] = resized
    return padded, sc, (pw, ph)

def decode(raw, grids, strides):
    p = raw[0].copy()
    p[:, :2] = (p[:, :2] + grids) * strides
    p[:, 2:4] = np.exp(p[:, 2:4]) * strides
    cx, cy, w, h = p[:,0], p[:,1], p[:,2], p[:,3]
    obj = p[:, 4:5]; cls = p[:, 5:]
    cid = cls.argmax(1); cc = cls[np.arange(len(cid)), cid]
    scores = obj.squeeze() * cc
    return np.stack([cx-w/2, cy-h/2, cx+w/2, cy+h/2, scores, cc, cid.astype(np.float32)], 1)

def nms(dets, iou_t):
    if len(dets) == 0: return dets
    x1,y1,x2,y2 = dets[:,0],dets[:,1],dets[:,2],dets[:,3]
    areas = (x2-x1)*(y2-y1); order = dets[:,4].argsort()[::-1]; keep = []
    while order.size > 0:
        i = order[0]; keep.append(i)
        if order.size == 1: break
        xx1=np.maximum(x1[i],x1[order[1:]]); yy1=np.maximum(y1[i],y1[order[1:]])
        xx2=np.minimum(x2[i],x2[order[1:]]); yy2=np.minimum(y2[i],y2[order[1:]])
        inter = np.maximum(0,xx2-xx1)*np.maximum(0,yy2-yy1)
        iou = inter/(areas[i]+areas[order[1:]]-inter+1e-6)
        order = order[1:][iou <= iou_t]
    return dets[keep]

def postproc(dets, sc, pad, st, nt):
    m = dets[:,4] > st; d = dets[m]
    if len(d)==0: return np.empty((0,7))
    d[:,0]=(d[:,0]-pad[0])/sc; d[:,1]=(d[:,1]-pad[1])/sc
    d[:,2]=(d[:,2]-pad[0])/sc; d[:,3]=(d[:,3]-pad[1])/sc
    final = [nms(d[d[:,6].astype(int)==c], nt) for c in np.unique(d[:,6].astype(int))]
    return np.concatenate(final, 0) if final else np.empty((0,7))

sys.path.insert(0, cfg["py_compute_dir"])
import py_compute as c

grids, strides = generate_yolox_grid(cfg["input_h"], cfg["input_w"], cfg["strides"])
qio = cfg["quantized_io"]
all_results = []

for img_path in cfg["sample_images"]:
    img = cv2.imread(img_path)
    if img is None: all_results.append(np.empty((0,7))); continue
    padded, sc, pad = letterbox_resize(img, cfg["input_w"], cfg["input_h"], cfg["pad_value"])
    blob = np.ascontiguousarray(padded.astype(np.float32)[np.newaxis, ...])
    if qio:
        blob = np.clip(np.round(blob / qio["input_scale"] + qio["input_zp"]), -128, 127).astype(np.int8)
    raw = c.compute(blob)[0]
    if qio:
        raw = (raw.astype(np.float32) - qio["output_zp"]) * qio["output_scale"]
    elif raw.dtype != np.float32:
        raw = raw.astype(np.float32)
    if raw.ndim == 2: raw = raw[np.newaxis, ...]
    dets = decode(raw, grids, strides)
    all_results.append(postproc(dets, sc, pad, cfg["score_thresh"], cfg["nms_thresh"]))

with open("{result_file}", "wb") as f:
    pickle.dump(all_results, f)
'''

    result = subprocess.run(
        [sys.executable, "-c", script],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"  ❌ Subprocess failed (exit code {result.returncode})")
        if result.stderr: print(f"  stderr:\n{result.stderr[:2000]}")
        return None

    with open(result_file, "rb") as f:
        return pickle.load(f)

# ── Run inference ─────────────────────────────────────────────────────────────
SAMPLE_DIR = OD_ROOT / "yolox_tiny" / "python" / "sample_images"
SAMPLE_IMAGES = sorted(SAMPLE_DIR.glob("*.jpg")) + sorted(SAMPLE_DIR.glob("*.JPEG")) + sorted(SAMPLE_DIR.glob("*.png"))
if not SAMPLE_IMAGES:
    raise FileNotFoundError(f"No sample images found in {SAMPLE_DIR}")

# TFLite INT8 reference
print(f"\nRunning TFLite INT8 reference ...")
tflite_int8_results = [run_tflite_yolox(INT8_PATH, cv2.imread(str(p))) for p in SAMPLE_IMAGES]

# MERA CPU py_compute
print(f"Running MERA CPU (py_compute) inference ...")
mera_results = run_mera_inference_subprocess(py_compute_dir, SAMPLE_IMAGES, quantized_io)

if mera_results is None:
    raise RuntimeError("MERA inference failed — see errors above.")

# ── Visualise side-by-side ────────────────────────────────────────────────────
for i, img_path in enumerate(SAMPLE_IMAGES):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue

    vis_tflite = draw_results(
        cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB),
        tflite_int8_results[i], COCO_CLASSES,
    )
    vis_mera = draw_results(
        cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB),
        mera_results[i], COCO_CLASSES,
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(vis_tflite)
    axes[0].set_title(f"TFLite INT8 — {len(tflite_int8_results[i])} det.")
    axes[0].axis("off")
    axes[1].imshow(vis_mera)
    axes[1].set_title(f"MERA CPU — {len(mera_results[i])} det.")
    axes[1].axis("off")
    plt.suptitle(img_path.name, fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"\n  {img_path.name}:")
    print(f"    TFLite INT8 : {len(tflite_int8_results[i])} detection(s)")
    for r in tflite_int8_results[i]:
        cname = COCO_CLASSES[int(r[6])]
        print(f"      {cname:<20s}  score={r[4]:.3f}  box=[{r[0]:.0f},{r[1]:.0f},{r[2]:.0f},{r[3]:.0f}]")
    print(f"    MERA CPU    : {len(mera_results[i])} detection(s)")
    for r in mera_results[i]:
        cname = COCO_CLASSES[int(r[6])]
        print(f"      {cname:<20s}  score={r[4]:.3f}  box=[{r[0]:.0f},{r[1]:.0f},{r[2]:.0f},{r[3]:.0f}]")

print(f"\n{'='*60}")
print(f"  ✅ MERA CPU model verification complete!")
print(f"{'='*60}")

---
### 📄 Understanding the Generated C Files

MERA produces C source files in `embedded_c/src_mcu_npu/` (NPU) or `embedded_c/src_mcu/` (CPU).
The key files are listed below:

| File | Purpose | Target |
|------|---------|--------|
| `model.c / model.h` | Top-level model entry point — orchestrates subgraph execution. | NPU only |
| `hal_entry.c` | Hardware Abstraction Layer — board-specific init (clocks, Ethos-U driver). | NPU only |
| `sub_XXXX_model_data.c/.h` | Vela-compiled command stream & weight data for NPU subgraphs. | NPU only |
| `sub_XXXX_tensors.c/.h` | Tensor arena allocation for NPU subgraphs. | NPU only |
| `sub_XXXX_command_stream.c/.h` | Ethos-U command stream blobs (compiled by Vela). | NPU only |
| `sub_XXXX_invoke.c/.h` | Invoke helpers that feed command streams to the Ethos-U driver. | NPU only |
| `compute_sub_0000.c/.h` | Main inference graph — kernel calls for each layer (NPU-accelerated or CMSIS-NN). | Both |
| `kernel_library_int.c/.h` | INT8/INT16 kernel implementations (Ethos-U offload or CMSIS-NN). | Both |
| `kernel_library_utils.c/.h` | Shared helpers — memory layout, tensor management, dequantization. | Both |

---

## 🎉 You're Done!

You have successfully completed the full pipeline from a pretrained YOLOX-Tiny model to MCU-ready C source files:

```
✅  Setup       — Configuration
✅  Section 1   — Download & build models from scratch (optional)
    └─ 1.1 — Verify original PyTorch model (baseline inference)
✅  Section 2   — Verify pre-built ONNX / TFLite models
✅  Section 3   — YOLOX preprocessing & postprocessing helpers
✅  Section 4   — Quick inference test (PyTorch vs FP32 vs INT8)
✅  Section 5   — MERA compilation → embedded C source files
    ├─ 5.1  NPU compilation (Ethos-U55)
    └─ 5.2  CPU-only compilation (CMSIS-NN) + host evaluation build
✅  Section 6   — Verify MERA-compiled CPU model (py_compute inference)
    └─ Compare detections vs TFLite INT8 reference
```

The generated C files in `embedded_c/src_mcu/` (and `src_mcu_npu/` for the NPU path) are ready to be integrated into your **e² studio** project and flashed to the **Renesas RA8P1** board.

---